# ДЗ Семинар №9. Транзакции, MVCC, изоляция и блокировки
# Выполнил Чесноков Данила ДЭ 16-25

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно.
4. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера**.
- Учётные данные БД: пользователь `postgres`, база `demo`. Если в вашем окружении включён пароль — добавьте `password="..."` в словарь `DB` ниже.
- Jupyter-пароль: `student`.

## План

1. Как устроена транзакционность: XID, CLOG, wraparound, freeze.
2. Компромисс консистентность vs доступность; MVCC.
3. Распределённые транзакции: 2PC и Saga.
4. Уровни изоляции в PostgreSQL.
5. Best practices по транзакциям.
6. Пессимистичные блокировки (`SELECT … FOR UPDATE`) и SQLAlchemy.
7. Оптимистичный контроль версий.
8. Deadlock и его диагностика.


In [19]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "demo",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
SQLALCHEMY_URL = (
    f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)

from sqlalchemy import create_engine, text
engine = create_engine(SQLALCHEMY_URL, echo=False)

print("\npsycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)

# Быстрая проверка подключения
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("\nPG version:", cur.fetchone()[0].split(" on ")[0])


Python: 3.11.17
psycopg     : 3.3.6
sqlalchemy  : 2.1.6
pandas      : 3.1.0
numpy       : 2.4.6

psycopg DSN      : host=localhost port=5432 dbname=demo user=postgres
SQLAlchemy URL   : postgresql+psycopg://postgres@localhost:5432/demo

PG version: PostgreSQL 15.19 (Debian 15.19-0+deb12u1)


## 2.1 Как устроена транзакционность в PostgreSQL

Когда транзакция выполняет первую **изменяющую** операцию (`INSERT/UPDATE/DELETE`) или явно запрашивает XID через `SELECT txid_current()`, PostgreSQL назначает ей уникальный **XID (Transaction ID)**. Пустой `BEGIN; SELECT 1; COMMIT;` **не получает XID вообще** — это важно, потому что 32-битное пространство XID конечно (~4 млрд значений).

- Каждая строка в heap имеет заголовок с `xmin` (XID вставившей транзакции) и `xmax` (XID удалившей/изменившей).
- Статус транзакции (IN_PROGRESS / COMMITTED / ABORTED / SUBCOMMITTED) хранится в **CLOG** — компактном массиве 2-битных состояний (каталог `pg_xact`).
- **`SELECT` без `FOR UPDATE` не блокирует и не блокируется `UPDATE`/`DELETE`** — читатель работает со снапшотом, а не с блокировками строк.

### Проблема wraparound и freeze

XID цикличен. Если старые строки не «заморозить», после переполнения счётчика они станут «из будущего» и пропадут из видимости. Autovacuum заменяет их на `FrozenTransactionId`, который всегда считается старым. Долгие транзакции мешают vacuum → риск wraparound и аварийной остановки.

**Золотое правило: транзакции должны быть короткими.**


In [20]:
# Демо: когда XID присваивается, а когда нет
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT txid_current_if_assigned()")
        print("XID в начале транзакции (до изменений):", cur.fetchone()[0])
    conn.commit()

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS xid_demo")
        cur.execute("CREATE TABLE xid_demo (id INT PRIMARY KEY, val INT)")
        cur.execute("INSERT INTO xid_demo VALUES (1, 100)")
        cur.execute("SELECT txid_current_if_assigned()")
        print("XID после INSERT:", cur.fetchone()[0])
    conn.commit()

# Проверим xmin/xmax у строки
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, val, xmin, xmax FROM xid_demo")
        print("\n(id, val, xmin, xmax):", cur.fetchone())

# Параметры autovacuum, связанные с freeze
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT name, setting FROM pg_settings
            WHERE name IN ('autovacuum', 'autovacuum_freeze_max_age')
        """)
        for name, val in cur.fetchall():
            print(f"{name:30s} = {val}")


XID в начале транзакции (до изменений): None
XID после INSERT: 829

(id, val, xmin, xmax): (1, 100, '829', '0')
autovacuum                     = on
autovacuum_freeze_max_age      = 200000000


### Упражнение 1. Разберитесь с XID и активными снапшотами

**Задание.** Продемонстрируйте, что `txid_current_if_assigned()` возвращает `NULL` до изменений, а после `INSERT` — реальный XID. Дополнительно посмотрите, что видно в `pg_stat_activity` по вашей сессии: `backend_xid` (XID, назначенный текущей транзакции) и `backend_xmin` (нижняя граница снапшота).

**Шаги.**
1. В новой транзакции выполните `SELECT txid_current_if_assigned()` — напечатайте результат.
2. Сделайте `INSERT` в `xid_demo` (используйте id 100).
3. Снова выполните `SELECT txid_current_if_assigned()` — значение должно быть не NULL.
4. Из **второго** соединения (`psycopg.connect(...)`) запросите `pg_stat_activity` по `pid = pg_backend_pid()` первой сессии: посмотрите `backend_xid`, `backend_xmin`.
5. Сделайте `COMMIT` в первой сессии и повторите п.4 — что изменилось?

**Критерий успеха.** До `INSERT` — `None`, после — число; `backend_xid` виден, пока транзакция открыта.


In [21]:
# УПРАЖНЕНИЕ 1. XID и снапшоты: txid_current_if_assigned() и pg_stat_activity
#
# Что делаем: в одной транзакции показываем, что XID выдаётся не на BEGIN, а только на первое
# изменение, и смотрим на эту транзакцию «снаружи» — из второго соединения через
# pg_stat_activity (колонки backend_xid и backend_xmin).
#
# Общая идея: XID — это «номер» транзакции, который нужен только тем транзакциям, которые что-то
# ПИШУТ (он попадает в xmin/xmax версий строк). Читающей транзакции XID не нужен, поэтому
# PostgreSQL его не тратит: пространство XID конечно (32 бита), и расходовать его зря нельзя.

def get_activity(pid):
    """Смотрит сессию pid ИЗ ДРУГОГО соединения. Возвращает (state, backend_xid, backend_xmin)."""
    # autocommit=True: мониторинговое соединение не держит собственную транзакцию. Это важно:
    # внутри одной транзакции PostgreSQL кеширует снимок статистики (pg_stat_activity), и
    # повторные запросы показывали бы устаревшие данные. С autocommit каждый запрос —
    # отдельная транзакция, значит, данные всегда свежие.
    # Остальные параметры по умолчанию: row_factory не задан -> строки приходят кортежами.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            # backend_xid и backend_xmin имеют тип xid, для которого у psycopg нет загрузчика
            # (пришла бы строка). Приводим xid -> text -> bigint, чтобы получить обычное число;
            # NULL при приведении остаётся NULL и превращается в Python None.
            cur.execute(
                "SELECT state, backend_xid::text::bigint, backend_xmin::text::bigint "
                "FROM pg_stat_activity WHERE pid = %s",
                (pid,),
            )
            return cur.fetchone()

# --- Подготовка (отдельная короткая транзакция) ----------------------------------
# Строку id=100 удаляем заранее, чтобы ячейку можно было запускать повторно: иначе второй
# запуск упал бы на дубликате первичного ключа. Это отдельная транзакция, она закрывается
# на выходе из with (COMMIT) ДО эксперимента и не влияет на проверяемую транзакцию.
with psycopg.connect(PSYCOPG_CONN) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS xid_demo (id INT PRIMARY KEY, val INT)")
    conn.execute("DELETE FROM xid_demo WHERE id = 100")

# --- Основной эксперимент: сессия №1 ----------------------------------------------
# psycopg.connect(PSYCOPG_CONN) без доп. параметров:
#  * autocommit=False (по умолчанию): перед первым запросом psycopg сам отправляет BEGIN,
#    транзакция живёт до commit()/rollback();
#  * изоляция — по умолчанию READ COMMITTED (настройка сервера). Для этого упражнения она не
#    принципиальна: мы смотрим на XID, а не на видимость данных.
with psycopg.connect(PSYCOPG_CONN) as conn1:
    with conn1.cursor() as cur1:

        # Шаг 1. XID до изменений.
        # txid_current_if_assigned() — «подглядывающая» функция: возвращает XID, если он уже
        # назначен, и NULL, если нет. В отличие от txid_current() она сама XID НЕ выдаёт
        # (txid_current() назначил бы его принудительно, и эксперимент потерял бы смысл).
        cur1.execute("SELECT txid_current_if_assigned()")
        xid_before = cur1.fetchone()[0]
        print("1) XID до изменений      :", xid_before)          # ожидаем None

        # Шаг 2. Первая изменяющая операция.
        # Параметры передаём отдельно от текста запроса (%s), как и в семинаре 8.
        # INSERT создаёт новую версию строки, в её xmin надо записать XID вставившей
        # транзакции — именно в этот момент PostgreSQL и выдаёт XID.
        cur1.execute("INSERT INTO xid_demo (id, val) VALUES (%s, %s)", (100, 1000))

        # Шаг 3. XID после изменения.
        cur1.execute("SELECT txid_current_if_assigned()")
        xid_after = cur1.fetchone()[0]
        print("3) XID после INSERT      :", xid_after)           # ожидаем число

        # pid нашего бэкенда — по нему второе соединение найдёт нас в pg_stat_activity.
        # (Тот же номер доступен без запроса: conn1.info.backend_pid.)
        cur1.execute("SELECT pg_backend_pid()")
        pid1 = cur1.fetchone()[0]
        print("   pid сессии 1          :", pid1)

        # Шаг 4. Смотрим на открытую транзакцию из второго соединения.
        state, b_xid, b_xmin = get_activity(pid1)
        print("\n4) Транзакция ОТКРЫТА (смотрим из соединения №2):")
        print(f"   state={state!r}, backend_xid={b_xid}, backend_xmin={b_xmin}")
        # txid_current_if_assigned() возвращает 64-битный номер (в старших 32 битах — «эпоха»
        # счётчика, сколько раз он переполнялся), а backend_xid — «короткий» 32-битный XID.
        # Поэтому сравниваем с остатком от деления на 2**32.
        print("   backend_xid совпал с XID из шага 3:", b_xid == xid_after % 2**32)

    # Шаг 5. COMMIT в первой сессии. Соединение при этом остаётся открытым (закроется на
    # выходе из внешнего with), поэтому сессию с тем же pid по-прежнему видно в pg_stat_activity.
    conn1.commit()
    state, b_xid, b_xmin = get_activity(pid1)
    print("\n5) Транзакция ЗАВЕРШЕНА (COMMIT):")
    print(f"   state={state!r}, backend_xid={b_xid}, backend_xmin={b_xmin}")

print("\nКритерий: до INSERT None ->", xid_before is None,
      "| после INSERT число ->", isinstance(xid_after, int))

# ЧТО ИЗМЕНИЛОСЬ ПОСЛЕ COMMIT:
# * state: 'idle in transaction' -> 'idle'. Это самый «неприятный» статус для мониторинга:
#   сессия ничего не делает, но транзакция открыта и держит ресурсы.
# * backend_xid: число -> NULL. XID принадлежит транзакции; она закончилась — XID у сессии нет.
#   (Сам XID не исчез: он записан в xmin вставленной строки, а в CLOG для него стоит COMMITTED.)
# * backend_xmin: на READ COMMITTED, как правило, NULL и до COMMIT. Снимок здесь создаётся на
#   КАЖДЫЙ оператор и освобождается сразу после него, поэтому в состоянии idle in transaction
#   нижней границы снимка нет. На REPEATABLE READ снимок один на всю транзакцию, и backend_xmin
#   держится до её конца — это и показывает упражнение 2.

1) XID до изменений      : None
3) XID после INSERT      : 830
   pid сессии 1          : 1687

4) Транзакция ОТКРЫТА (смотрим из соединения №2):
   state='idle in transaction', backend_xid=830, backend_xmin=None
   backend_xid совпал с XID из шага 3: True

5) Транзакция ЗАВЕРШЕНА (COMMIT):
   state='idle', backend_xid=None, backend_xmin=None

Критерий: до INSERT None -> True | после INSERT число -> True


## 2.2 Компромисс: консистентность vs доступность/производительность

Строгая изоляция требует координации, координация снижает пропускную способность. PostgreSQL выбрал **MVCC (Multi-Version Concurrency Control)**: читатели не блокируют писателей, создаются новые версии строк.

Плюсы MVCC:
- высокая пропускная способность чтения;
- читатели не видят «грязных» данных;
- нет read-блокировок.

Минусы MVCC:
- нужен периодический **VACUUM** (autovacuum) для очистки мёртвых версий;
- **bloat** таблиц при долгих транзакциях;
- риск wraparound, если vacuum не успевает.

**Ключевая мысль:** долгая транзакция держит старый снапшот и мешает vacuum удалить старые версии. Поэтому её влияние выходит за рамки «моей» таблицы — она бьёт по всей БД.


In [22]:
# Демо: активная транзакция держит backend_xmin
with psycopg.connect(PSYCOPG_CONN) as conn1:
    with conn1.cursor() as cur1:
        cur1.execute("BEGIN ISOLATION LEVEL REPEATABLE READ")
        cur1.execute("SELECT count(*) FROM xid_demo")  # создаёт snapshot
        cur1.fetchone()
        pid1 = None
        cur1.execute("SELECT pg_backend_pid()")
        pid1 = cur1.fetchone()[0]
        print("Сессия 1 pid =", pid1)

        # Из второго соединения смотрим pg_stat_activity
        with psycopg.connect(PSYCOPG_CONN) as conn2:
            with conn2.cursor() as cur2:
                cur2.execute(
                    "SELECT pid, state, backend_xid, backend_xmin "
                    "FROM pg_stat_activity WHERE pid = %s", (pid1,)
                )
                print("\nПока транзакция 1 открыта:")
                print(cur2.fetchone())
    conn1.rollback()

# Транзакция закрыта — backend_xmin освобождён
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT pid, state, backend_xid, backend_xmin "
            "FROM pg_stat_activity WHERE pid = %s", (pid1,)
        )
        print("\nПосле rollback:")
        print(cur.fetchone())


Сессия 1 pid = 1690

Пока транзакция 1 открыта:
(1690, 'idle in transaction', None, '831')

После rollback:
None


### Упражнение 2. Долгая транзакция мешает vacuum

**Задание.** Покажите, что `backend_xmin` держится всё время жизни транзакции и освобождается только после её завершения. Это и есть причина, по которой долгие транзакции мешают vacuum.

**Шаги.**
1. Откройте соединение `conn1`, начните `BEGIN ISOLATION LEVEL REPEATABLE READ` и выполните `SELECT` из `xid_demo` — создайте snapshot.
2. Из второго соединения посмотрите `pg_stat_activity` по `pid` первой сессии: `backend_xmin` должен быть не NULL.
3. Сделайте в первой сессии ещё один `SELECT` из `xid_demo` — `backend_xmin` не меняется (snapshot стабилен).
4. Закройте транзакцию `ROLLBACK` и снова проверьте `pg_stat_activity` — `backend_xmin` должен стать NULL.
5. В комментарии к коду напишите 1–2 строки: почему удержание `backend_xmin` блокирует очистку мёртвых версий?

**Критерий успеха.** В консоли видно `backend_xmin: <число>` пока транзакция открыта, и `None` после закрытия.


In [23]:
# УПРАЖНЕНИЕ 2. Долгая транзакция держит backend_xmin и тем самым мешает VACUUM
#
# Что делаем: открываем транзакцию на REPEATABLE READ, создаём снимок и из другого соединения
# проверяем, что backend_xmin не NULL и не меняется, пока транзакция жива; после ROLLBACK он
# освобождается.
from psycopg import IsolationLevel

def get_activity(pid):
    """Состояние сессии pid из ДРУГОГО соединения: (state, backend_xid, backend_xmin).
    Та же функция, что в упражнении 1; продублирована, чтобы ячейка запускалась отдельно."""
    # autocommit=True — иначе pg_stat_activity внутри долгой транзакции мониторинга
    # показывал бы закешированные (устаревшие) данные.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT state, backend_xid::text::bigint, backend_xmin::text::bigint "
                "FROM pg_stat_activity WHERE pid = %s",
                (pid,),
            )
            return cur.fetchone()

def burn_xid():
    """Сдвигает глобальный счётчик XID вперёд: имитирует «жизнь» остальной системы."""
    # txid_current() назначает XID ТЕКУЩЕЙ транзакции принудительно. Соединение в autocommit,
    # поэтому транзакция тут же завершается, и XID оказывается израсходован.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        return conn.execute("SELECT txid_current()").fetchone()[0]

# Таблица xid_demo создана в демо выше; на случай запуска ячейки отдельно:
with psycopg.connect(PSYCOPG_CONN) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS xid_demo (id INT PRIMARY KEY, val INT)")

with psycopg.connect(PSYCOPG_CONN) as conn1:
    # ВАЖНО: уровень изоляции задаём СВОЙСТВОМ соединения, ДО первого запроса.
    # Почему не cur.execute("BEGIN ISOLATION LEVEL REPEATABLE READ"), как в демо выше?
    # При autocommit=False psycopg сам отправляет BEGIN перед первым запросом (в том числе перед
    # нашим «BEGIN ...»). Сервер отвечает предупреждением «there is already a transaction in
    # progress» и игнорирует второй BEGIN, то есть уровень остаётся READ COMMITTED — а там
    # снимок не держится между операторами, и backend_xmin в простое был бы NULL.
    # Свойство isolation_level заставляет psycopg отправить правильный
    # «BEGIN ISOLATION LEVEL REPEATABLE READ». Менять его можно только вне транзакции
    # (сейчас соединение только что открыто — всё в порядке).
    conn1.isolation_level = IsolationLevel.REPEATABLE_READ

    with conn1.cursor() as cur1:
        # Шаг 1. Первый SELECT создаёт снимок. На REPEATABLE READ снимок создаётся не на BEGIN,
        # а на первом запросе, который читает данные, и живёт до конца транзакции.
        cur1.execute("SELECT count(*) FROM xid_demo")
        n_first = cur1.fetchone()[0]

        # Проверяем, что уровень действительно применился (SHOW снимок не создаёт).
        cur1.execute("SHOW transaction_isolation")
        print("Уровень изоляции сессии 1:", cur1.fetchone()[0])      # repeatable read

        cur1.execute("SELECT pg_backend_pid()")
        pid1 = cur1.fetchone()[0]

        # Шаг 2. Смотрим из соединения №2: backend_xmin должен быть числом.
        state, b_xid, xmin_1 = get_activity(pid1)
        print(f"\n2) После первого SELECT : state={state!r}, backend_xid={b_xid}, backend_xmin={xmin_1}")

        # Двигаем счётчик XID вперёд на несколько номеров: «мир» продолжает жить, коммитятся
        # другие транзакции. Если бы xmin мог «ехать» вместе с ними, мы бы это увидели.
        last_xid = [burn_xid() for _ in range(3)][-1]
        print(f"   (глобальный счётчик продвинули до txid={last_xid})")

        # Шаг 3. Второй SELECT в той же транзакции: снимок стабилен, xmin не меняется.
        cur1.execute("SELECT count(*) FROM xid_demo")
        n_second = cur1.fetchone()[0]
        state, b_xid, xmin_2 = get_activity(pid1)
        print(f"3) После второго SELECT : state={state!r}, backend_xid={b_xid}, backend_xmin={xmin_2}")
        print("   backend_xmin не изменился:", xmin_1 == xmin_2,
              "| count(*) в обоих запросах одинаков:", n_first == n_second)

    # Шаг 4. ROLLBACK: транзакция завершена, снимок отпущен.
    # (conn1.rollback() вместо commit(): транзакция только читала, фиксировать нечего.)
    conn1.rollback()
    state, b_xid, xmin_3 = get_activity(pid1)
    print(f"4) После ROLLBACK       : state={state!r}, backend_xid={b_xid}, backend_xmin={xmin_3}")

print("\nКритерий: во время транзакции xmin =", xmin_1, "| после закрытия xmin =", xmin_3)

# ПОЧЕМУ УДЕРЖАНИЕ backend_xmin БЛОКИРУЕТ ОЧИСТКУ МЁРТВЫХ ВЕРСИЙ:
# 1) Мёртвую версию строки (её удалили или обновили) VACUUM может физически убрать, только если
#    она не видна НИ ОДНОМУ существующему снимку. Транзакция с backend_xmin = N говорит: «я могу
#    обратиться к любой версии, которая была актуальна на момент N», — поэтому все версии,
#    удалённые транзакциями с номером >= N, удалять нельзя.
# 2) VACUUM берёт самый старый backend_xmin среди всех сессий (xmin horizon) и чистит только
#    то, что умерло раньше этой границы. Одна забытая транзакция тормозит очистку у ВСЕХ
#    таблиц базы, а не только у тех, которые она читала: таблицы пухнут (bloat), индексы тоже,
#    а заморозка (freeze) старых XID откладывается, что приближает wraparound.
# Именно поэтому правило «транзакции должны быть короткими» — не вопрос стиля.

Уровень изоляции сессии 1: repeatable read

2) После первого SELECT : state='idle in transaction', backend_xid=None, backend_xmin=831
   (глобальный счётчик продвинули до txid=833)
3) После второго SELECT : state='idle in transaction', backend_xid=None, backend_xmin=831
   backend_xmin не изменился: True | count(*) в обоих запросах одинаков: True
4) После ROLLBACK       : state='idle', backend_xid=None, backend_xmin=None

Критерий: во время транзакции xmin = 831 | после закрытия xmin = None


## 2.3 Транзакции в распределённых системах

Когда данные размазаны по нескольким базам (микросервисам) или шардам, классические ACID-транзакции «из коробки» не работают. Два основных подхода:

### 2PC (Two-Phase Commit)

- **Фаза 1 (Prepare):** координатор спрашивает все узлы: «Готовы закоммитить?». Узлы блокируют данные и отвечают «Да/Нет».
- **Фаза 2 (Commit/Rollback):** если все «Да» — `COMMIT`, иначе `ROLLBACK`.
- **Минус:** падение координатора после Prepare оставляет участников в заблокированном состоянии → страдает доступность.
- PostgreSQL поддерживает 2PC через `PREPARE TRANSACTION` / `COMMIT PREPARED` / `ROLLBACK PREPARED` (требуется `max_prepared_transactions > 0`).

### Saga + Eventual Consistency

- Вместо одной распределённой транзакции — цепочка локальных.
- При ошибке шага N запускаются компенсирующие транзакции (отменяющие 1…N-1).
- Данные временно неконсистентны, но система Available. Стандарт для высоконагруженных микросервисов.
- **Оркестрация** — центральный координатор. **Хореография** — сервисы подписываются на события друг друга через брокер (Kafka, RabbitMQ).


In [24]:
# Демо: PREPARE TRANSACTION (2PC) в PostgreSQL
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS twopc_demo")
        cur.execute("CREATE TABLE twopc_demo (id INT PRIMARY KEY, val INT)")
        cur.execute("INSERT INTO twopc_demo VALUES (1, 100)")
        cur.execute("SHOW max_prepared_transactions")
        max_prep = int(cur.fetchone()[0])
        print("max_prepared_transactions =", max_prep)
        if max_prep == 0:
            print("PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0 и перезапуск PG).")
        else:
            # Явно готовим транзакцию
            cur.execute("BEGIN")
            cur.execute("UPDATE twopc_demo SET val = 200 WHERE id = 1")
            cur.execute("PREPARE TRANSACTION 'demo_prepared'")
            print("Prepared.")

# Смотрим список подготовленных транзакций
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT gid, prepared, owner, database FROM pg_prepared_xacts")
        rows = cur.fetchall()
        print("pg_prepared_xacts:", rows)

# Финализируем (в реальной системе это делает координатор)
if rows:
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute("COMMIT PREPARED 'demo_prepared'")
            cur.execute("SELECT val FROM twopc_demo WHERE id = 1")
            print("After COMMIT PREPARED: val =", cur.fetchone()[0])


max_prepared_transactions = 0
PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0 и перезапуск PG).
pg_prepared_xacts: []


### Упражнение 3. PREPARE TRANSACTION и откат

**Задание.** Проведите полный цикл 2PC-транзакции: подготовьте, посмотрите в `pg_prepared_xacts`, откатите через `ROLLBACK PREPARED`, убедитесь, что изменения не применились.

**Шаги.**
1. Проверьте `SHOW max_prepared_transactions`. Если `0` — выведите сообщение, что демо пропущено.
2. В `twopc_demo` (id=2) вставьте значение `500`.
3. Подготовьте транзакцию: `BEGIN; INSERT ...; PREPARE TRANSACTION 'ex3_prepared'`.
4. Из другого соединения посмотрите `pg_prepared_xacts` — ваша транзакция должна быть в списке.
5. Выполните `ROLLBACK PREPARED 'ex3_prepared'`.
6. Проверьте `SELECT * FROM twopc_demo WHERE id = 2` — пусто (изменения откатились).

**Критерий успеха.** В `pg_prepared_xacts` есть `ex3_prepared` до `ROLLBACK PREPARED`; после отката `SELECT` пуст.


In [25]:
# УПРАЖНЕНИЕ 3. Полный цикл 2PC: PREPARE TRANSACTION -> pg_prepared_xacts -> ROLLBACK PREPARED
#
# Что делаем: готовим транзакцию к фиксации (фаза 1 двухфазного коммита), убеждаемся, что она
# «подвисла» в pg_prepared_xacts и не зависит от соединения, и откатываем её (фаза 2).
from psycopg import sql
from psycopg.errors import LockNotAvailable

GID = "ex3_prepared"   # global transaction id: имя, по которому «координатор» найдёт транзакцию

def my_prepared():
    """Возвращает строки pg_prepared_xacts для нашей транзакции (из НОВОГО соединения)."""
    # autocommit=True: ROLLBACK PREPARED и PREPARE TRANSACTION нельзя выполнять внутри блока
    # транзакции, а просто читать pg_prepared_xacts удобнее без неявного BEGIN.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        return conn.execute(
            "SELECT gid, prepared, owner, database FROM pg_prepared_xacts "
            "WHERE gid = %s AND database = current_database()",
            (GID,),
        ).fetchall()

def exercise3():
    # Шаг 1. Проверяем, разрешены ли подготовленные транзакции вообще.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        max_prep = int(conn.execute("SHOW max_prepared_transactions").fetchone()[0])
    print("max_prepared_transactions =", max_prep)
    if max_prep == 0:
        # По умолчанию 0: параметр меняется только в postgresql.conf с перезапуском сервера.
        print("Демо пропущено: PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0).")
        return

    # --- Подготовка: делаем ячейку перезапускаемой -------------------------------
    # Если прошлый запуск упал между PREPARE и ROLLBACK PREPARED, транзакция с таким gid
    # осталась бы в системе (подготовленные транзакции переживают закрытие соединения и даже
    # перезапуск сервера) и второй PREPARE упал бы с «transaction identifier already in use».
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        if my_prepared():
            # PREPARE TRANSACTION / ROLLBACK PREPARED — служебные команды и не принимают
            # параметры (%s), а имя нельзя подставлять f-строкой, поэтому собираем запрос через
            # sql.SQL + sql.Literal: драйвер сам корректно экранирует значение как литерал.
            conn.execute(sql.SQL("ROLLBACK PREPARED {}").format(sql.Literal(GID)))
            print("Убрали «хвост» от предыдущего запуска.")
        conn.execute("CREATE TABLE IF NOT EXISTS twopc_demo (id INT PRIMARY KEY, val INT)")
        conn.execute("DELETE FROM twopc_demo WHERE id = 2")

    # Шаги 2-3. Готовим транзакцию.
    # autocommit=True: psycopg НЕ открывает транзакцию сам, поэтому BEGIN пишем вручную —
    # так мы полностью управляем границами. (При autocommit=False ручной BEGIN дал бы
    # «there is already a transaction in progress».)
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        conn.execute("BEGIN")
        conn.execute("INSERT INTO twopc_demo (id, val) VALUES (%s, %s)", (2, 500))
        # PREPARE TRANSACTION: фаза 1. PostgreSQL сбрасывает состояние транзакции на диск и
        # «отрывает» её от этой сессии. Дальше ни COMMIT, ни ROLLBACK в ней выполнить нельзя:
        # транзакцию завершит команда COMMIT PREPARED / ROLLBACK PREPARED — из любой сессии.
        conn.execute(sql.SQL("PREPARE TRANSACTION {}").format(sql.Literal(GID)))
    print("Транзакция подготовлена, соединение, которое её создало, уже закрыто.")

    # Шаг 4. Смотрим из ДРУГОГО соединения: транзакция живёт сама по себе.
    rows = my_prepared()
    print("pg_prepared_xacts:", rows)
    print("   ex3_prepared в списке:", len(rows) == 1)

    # Пока транзакция не завершена, её изменения никому не видны...
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        visible = conn.execute("SELECT * FROM twopc_demo WHERE id = 2").fetchall()
        print("   строка id=2 видна другим сессиям до фазы 2:", visible)    # []

        # ...и она ДЕРЖИТ блокировки. Параллельная вставка того же ключа должна ждать решения
        # судьбы нашей транзакции. Чтобы ячейка не зависла навсегда, ограничиваем ожидание:
        # lock_timeout = 1s (без него вставка ждала бы, пока кто-то не сделает COMMIT/ROLLBACK).
        conn.execute("SET lock_timeout = '1s'")
        try:
            conn.execute("INSERT INTO twopc_demo (id, val) VALUES (%s, %s)", (2, 1))
            print("   (неожиданно) вставка прошла")
        except LockNotAvailable as e:
            print("   параллельная вставка id=2 ждёт и срывается по таймауту:", type(e).__name__)
            # Это и есть главный минус 2PC: после PREPARE участник отвечает «да» и обязан
            # держать ресурсы, пока координатор не примет решение. Если координатор пропал —
            # блокировки висят, и страдает доступность.

    # Шаг 5. Фаза 2: решение «откат» принимает «координатор» — здесь это просто другое соединение.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        conn.execute(sql.SQL("ROLLBACK PREPARED {}").format(sql.Literal(GID)))
    print("\nROLLBACK PREPARED выполнен. В pg_prepared_xacts осталось:", my_prepared())   # []

    # Шаг 6. Изменения не применились.
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        rows2 = conn.execute("SELECT * FROM twopc_demo WHERE id = 2").fetchall()
    print("SELECT * FROM twopc_demo WHERE id = 2 ->", rows2, "| пусто:", rows2 == [])

exercise3()

# ВАЖНО НА ПРАКТИКЕ: забытая подготовленная транзакция — тихая мина. Она держит блокировки,
# удерживает xmin (мешает VACUUM, см. упражнение 2) и переживает перезапуск сервера. Поэтому
# pg_prepared_xacts нужно мониторить, а «осиротевшие» транзакции — завершать вручную.

max_prepared_transactions = 0
Демо пропущено: PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0).


## 2.4 Уровни изоляции в PostgreSQL

Аномалии чтения:

| Аномалия | Описание |
|---|---|
| **Dirty Read** | Чтение данных, изменённых ещё не зафиксированной транзакцией |
| **Non-repeatable Read** | Повторный `SELECT` той же строки возвращает другое значение |
| **Phantom Read** | Повторный запрос по диапазону возвращает другой набор строк |
| **Lost Update** | Две транзакции читают одно значение, обе пишут — одна затирает другую |

Уровни изоляции **в реализации PostgreSQL**:

1. **READ UNCOMMITTED** — автоматически повышается до `READ COMMITTED`. Никогда не используется.
2. **READ COMMITTED** (default) — свежий snapshot на каждый **оператор**.
   - Закрывает: Dirty Read.
   - **Не** закрывает: Non-repeatable Read, Phantom Read, Lost Update.
3. **REPEATABLE READ** — один snapshot на всю транзакцию (создаётся на первом запросе, не на `BEGIN`).
   - Закрывает: Dirty Read, Non-repeatable Read, **Phantom Read** (отличие от стандарта SQL: в PG phantom закрыт).
   - **Не** закрывает: Lost Update.
   - При попытке изменить строку, изменённую параллельной транзакцией после snapshot’а: `ERROR: could not serialize access due to concurrent update`.
4. **SERIALIZABLE** — SSI (Serializable Snapshot Isolation): отслеживает read/write-зависимости, откатывает при обнаружении цикла.
   - Закрывает всё, включая Lost Update и Write Skew.
   - Самая дорогая по CPU/памяти; используйте там, где действительно нужна полная сериализуемость.

**Шпаргалка:**

| Уровень | Dirty Read | Non-repeatable | Phantom | Lost Update |
|---|---|---|---|---|
| READ UNCOMMITTED (PG) | ✗ | ✓ | ✓ | ✓ |
| READ COMMITTED | ✗ | ✓ | ✓ | ✓ |
| REPEATABLE READ | ✗ | ✗ | ✗ | ✓ |
| SERIALIZABLE | ✗ | ✗ | ✗ | ✗ |

(✗ — аномалия невозможна, ✓ — возможна)


In [26]:
# Демо: установка уровня изоляции и проверка
# Уровень по умолчанию
with engine.connect() as conn:
    iso = conn.execute(text("SHOW transaction_isolation")).scalar()
    print("Default isolation:", iso)

# Через SQLAlchemy execution_options
for level in ["READ COMMITTED", "REPEATABLE READ", "SERIALIZABLE"]:
    with engine.connect().execution_options(isolation_level=level) as conn:
        iso = conn.execute(text("SHOW transaction_isolation")).scalar()
        print(f"  requested={level!r:20s} -> actual={iso!r}")


Default isolation: read committed
  requested='READ COMMITTED'     -> actual='read committed'
  requested='REPEATABLE READ'    -> actual='repeatable read'
  requested='SERIALIZABLE'       -> actual='serializable'


### Упражнение 4. Non-repeatable Read на READ COMMITTED

**Задание.** Воспроизведите Non-repeatable Read на `READ COMMITTED` и убедитесь, что `REPEATABLE READ` её закрывает.

**Шаги.**
1. Создайте таблицу `ex4_accounts (id INT PRIMARY KEY, balance INT)` и вставьте `(1, 1000)`.
2. В двух потоках (`threading.Thread`):
   - **Поток A** открывает транзакцию с `READ COMMITTED`, читает `balance` (1000), спит 1 сек, читает ещё раз, печатает оба значения, делает `COMMIT`.
   - **Поток B** спит 0.3 сек, делает `UPDATE ... SET balance = 500 WHERE id = 1`, `COMMIT`.
3. Запустите и убедитесь, что второй `SELECT` в потоке A вернул **500** (non-repeatable read).
4. Повторите эксперимент, установив в потоке A `REPEATABLE READ`. Второй `SELECT` должен вернуть **1000**.
5. В комментарии: почему `REPEATABLE READ` закрывает эту аномалию?

**Критерий успеха.** В первом прогоне второе значение = 500, во втором = 1000.


In [27]:
# УПРАЖНЕНИЕ 4. Non-repeatable Read на READ COMMITTED и его отсутствие на REPEATABLE READ
#
# Что делаем: поток A дважды читает один и тот же баланс с паузой между чтениями, а поток B
# в паузе меняет баланс и фиксирует изменение. Повторяем эксперимент на двух уровнях изоляции.
import threading, time

# --- Подготовка таблицы ---------------------------------------------------------
# engine.begin() — одна транзакция: COMMIT при нормальном выходе, ROLLBACK при исключении.
# Уровень изоляции по умолчанию — READ COMMITTED; для DDL без конкурентов он ни на что не влияет.
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ex4_accounts"))
    conn.execute(text("CREATE TABLE ex4_accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO ex4_accounts VALUES (1, 1000)"))

def reader(isolation_level, out, first_read_done):
    """Поток A: читает баланс, спит 1 с, читает снова. Результат кладёт в словарь out."""
    # engine.connect().execution_options(isolation_level=...) — задаёт уровень изоляции для
    # ЭТОГО соединения: SQLAlchemy перед началом транзакции выставляет его у драйвера.
    # Менять уровень можно только до первого запроса (транзакция в SQLAlchemy 2.x
    # начинается неявно на первом execute). Когда соединение вернётся в пул, SQLAlchemy
    # сбросит уровень в значение по умолчанию, и «заражённое» соединение не уйдёт другим потокам.
    # Блок `with` на выходе закрывает соединение (возвращает в пул); всё, что не закоммичено,
    # откатывается — у engine.connect() в SQLAlchemy 2.x автокоммита нет.
    with engine.connect().execution_options(isolation_level=isolation_level) as conn:
        # SHOW снимка не создаёт, но подтверждает, что уровень применился.
        iso = conn.execute(text("SHOW transaction_isolation")).scalar()

        # Первое чтение. На REPEATABLE READ именно здесь создаётся снимок на всю транзакцию.
        first = conn.execute(text("SELECT balance FROM ex4_accounts WHERE id = 1")).scalar()
        print(f"  A [{iso}]: первое чтение  = {first}")
        first_read_done.set()          # сообщаем потоку B: «первое чтение сделано, можно писать»

        time.sleep(1)                  # окно, в которое B успевает изменить и зафиксировать данные

        second = conn.execute(text("SELECT balance FROM ex4_accounts WHERE id = 1")).scalar()
        print(f"  A [{iso}]: второе чтение  = {second}")
        conn.commit()                  # транзакция только читала, фиксируем для порядка
        out.update(iso=iso, first=first, second=second)

def writer(first_read_done):
    """Поток B: через 0.3 с меняет баланс на 500 и фиксирует."""
    time.sleep(0.3)
    # Страховка от гонки: ждём событие от A (до 5 с). Без неё, если подключение A окажется
    # медленнее 0.3 с, B мог бы обновить данные РАНЬШЕ первого чтения, и эксперимент
    # ничего бы не показал. Само событие на логику не влияет: A всё равно спит 1 с.
    first_read_done.wait(timeout=5)
    # UPDATE строки, которую A только читает, не блокируется: читатели в MVCC не держат
    # блокировок на строках, поэтому B не ждёт A.
    with engine.begin() as conn:
        conn.execute(text("UPDATE ex4_accounts SET balance = 500 WHERE id = 1"))
    print("  B: UPDATE balance = 500 выполнен и зафиксирован (COMMIT)")

def run_experiment(level):
    # Каждый прогон начинаем с исходного значения 1000.
    with engine.begin() as conn:
        conn.execute(text("UPDATE ex4_accounts SET balance = 1000 WHERE id = 1"))
    out = {}
    first_read_done = threading.Event()
    ta = threading.Thread(target=reader, args=(level, out, first_read_done))
    tb = threading.Thread(target=writer, args=(first_read_done,))
    ta.start(); tb.start()
    ta.join();  tb.join()          # join() — ждём завершения обоих потоков
    return out

# --- Прогон 1: READ COMMITTED -----------------------------------------------------
print("Прогон 1: READ COMMITTED")
res_rc = run_experiment("READ COMMITTED")
print("  итог:", (res_rc.get("first"), res_rc.get("second")),
      "-> non-repeatable read:", res_rc.get("second") == 500, "\n")

# --- Прогон 2: REPEATABLE READ ----------------------------------------------------
print("Прогон 2: REPEATABLE READ")
res_rr = run_experiment("REPEATABLE READ")
print("  итог:", (res_rr.get("first"), res_rr.get("second")),
      "-> чтения совпали:", res_rr.get("second") == 1000)

print("\nКритерий: прогон 1 — второе значение 500:", res_rc.get("second") == 500,
      "| прогон 2 — второе значение 1000:", res_rr.get("second") == 1000)

# ПОЧЕМУ REPEATABLE READ ЗАКРЫВАЕТ ЭТУ АНОМАЛИЮ:
# Видимость в PostgreSQL определяется снимком (snapshot): версия строки видна, если вставившая
# её транзакция (xmin) зафиксирована ДО снимка и не удалена (xmax) транзакцией, зафиксированной
# до снимка.
#  * READ COMMITTED берёт НОВЫЙ снимок на каждый оператор. Второй SELECT потока A получает
#    снимок уже после COMMIT потока B, видит новую версию строки (balance=500) и возвращает
#    другое значение, чем первый SELECT. Это и есть non-repeatable read.
#  * REPEATABLE READ берёт ОДИН снимок на первом запросе и использует его до конца транзакции.
#    Версия строки, созданная B после этого момента, для снимка A «из будущего» и невидима;
#    A продолжает видеть старую версию (balance=1000), которую MVCC для этого и хранит.
# Данные в БД при этом одинаковы в обоих прогонах (500 после B): отличается только то,
# какую версию видит читатель.

Прогон 1: READ COMMITTED
  A [read committed]: первое чтение  = 1000
  B: UPDATE balance = 500 выполнен и зафиксирован (COMMIT)
  A [read committed]: второе чтение  = 500
  итог: (1000, 500) -> non-repeatable read: True 

Прогон 2: REPEATABLE READ
  A [repeatable read]: первое чтение  = 1000
  B: UPDATE balance = 500 выполнен и зафиксирован (COMMIT)
  A [repeatable read]: второе чтение  = 1000
  итог: (1000, 1000) -> чтения совпали: True

Критерий: прогон 1 — второе значение 500: True | прогон 2 — второе значение 1000: True


## 2.5 Best practices по транзакциям

1. **Короткие транзакции.** Пока транзакция открыта, autovacuum не может очистить старые версии строк. Возникает bloat и растёт риск wraparound.
2. **Единый порядок блокировок.** Если обновляете несколько строк/таблиц — всегда в одном и том же порядке. Иначе почти гарантированно получите deadlock (см. раздел 5).
3. **Аналитика (OLAP) — не в OLTP-базе.**
   - Идеально — вынести на streaming-реплику.
   - Если всё же на основной: `BEGIN ISOLATION LEVEL SERIALIZABLE READ ONLY DEFERRABLE;` — PG отложит старт, чтобы не конфликтовать с параллельными SERIALIZABLE.
4. **Явные блокировки — точечно.** `FOR UPDATE` превращает optimistic concurrency в pessimistic; применяйте только там, где действительно нужно (списание денег, инвентарь, бронирование).
5. **Обрабатывайте serialization failures.** На `REPEATABLE READ` / `SERIALIZABLE` приложение **обязано** уметь повторять транзакцию при ошибке.


In [28]:
# Демо: диагностика долгих транзакций
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT pid,
                   now() - xact_start AS duration,
                   state,
                   left(query, 60) AS query_head
            FROM pg_stat_activity
            WHERE xact_start IS NOT NULL
              AND backend_type = 'client backend'
              AND pid <> pg_backend_pid()
            ORDER BY duration DESC NULLS LAST
            LIMIT 10
        """)
        rows = cur.fetchall()
        print("Долгие транзакции (pid, duration, state, query):")
        for r in rows:
            print(" ", r)
        if not rows:
            print("  (нет других активных транзакций)")


Долгие транзакции (pid, duration, state, query):
  (нет других активных транзакций)


### Упражнение 5. Диагностика долгих транзакций

**Задание.** Напишите запрос, который находит все транзакции, открытые дольше 1 секунды, и показывает их `pid`, длительность, `state`, `backend_xmin` и начало запроса.

**Шаги.**
1. Напишите SQL с фильтром `now() - xact_start > interval '1 second'`.
2. Выполните его через `psycopg` — напечатайте все строки.
3. Создайте в соседнем соединении долгую транзакцию (`BEGIN REPEATABLE READ; SELECT 1;` и подержите её через `time.sleep(2)`), чтобы проверить, что запрос её находит.
4. В комментарии: какие ещё поля `pg_stat_activity` полезны для диагностики блокировок?

**Критерий успеха.** Ваш «долгий» `SELECT` из п.3 попадает в вывод.


In [29]:
# УПРАЖНЕНИЕ 5. Диагностика долгих транзакций через pg_stat_activity
#
# Что делаем: пишем запрос, который находит транзакции старше 1 секунды, и проверяем его на
# искусственной «долгой» транзакции, открытой в фоновом потоке.
import time, threading
from psycopg import IsolationLevel

# Шаг 1. SQL-запрос.
#  * xact_start — момент начала ТЕКУЩЕЙ транзакции сессии; NULL, если транзакции нет
#    (сессия в состоянии idle), поэтому условие IS NOT NULL отсекает такие сессии;
#  * now() - xact_start > interval '1 second' — «возраст» транзакции больше секунды;
#  * backend_type = 'client backend' — только клиентские сессии, без служебных процессов
#    (autovacuum, checkpointer и т. п.);
#  * pid <> pg_backend_pid() — исключаем саму диагностическую сессию;
#  * backend_xmin::text — xid приводим к тексту, иначе psycopg вернёт его «сырым»;
#  * left(query, 60) — начало текста запроса (в state 'idle in transaction' там лежит
#    ПОСЛЕДНИЙ выполненный запрос сессии, что тоже полезно для поиска виновника).
LONG_TX_SQL = """
    SELECT pid,
           now() - xact_start       AS duration,
           state,
           backend_xmin::text       AS backend_xmin,
           left(query, 60)          AS query_head
    FROM pg_stat_activity
    WHERE xact_start IS NOT NULL
      AND now() - xact_start > interval '1 second'
      AND backend_type = 'client backend'
      AND pid <> pg_backend_pid()
    ORDER BY duration DESC
"""

def find_long_transactions():
    """Выполняет диагностический запрос и возвращает список строк."""
    # autocommit=True: now() в PostgreSQL — время НАЧАЛА текущей транзакции. Если бы
    # мониторинговое соединение держало одну транзакцию, now() «застыл» бы, и возраст чужих
    # транзакций считался бы неверно. В autocommit каждый запрос — своя транзакция.
    # (Альтернатива — clock_timestamp(): оно всегда возвращает реальное текущее время.)
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(LONG_TX_SQL)
            return cur.fetchall()

def print_rows(rows):
    if not rows:
        print("   (долгих транзакций нет)")
    for r in rows:
        print("  ", r)

# Шаг 2. Выполняем запрос «вхолостую»: пока долгих транзакций у нас нет.
print("До запуска долгой транзакции:")
print_rows(find_long_transactions())

# Шаг 3. Долгая транзакция в фоновом потоке.
def long_transaction(info, started, release):
    # psycopg.connect(...) — свой поток = своё соединение (соединения между потоками
    # не делят). Уровень REPEATABLE READ задаём свойством ДО первого запроса (см. упр. 2):
    # так снимок и backend_xmin держатся до конца транзакции.
    with psycopg.connect(PSYCOPG_CONN) as conn:
        conn.isolation_level = IsolationLevel.REPEATABLE_READ
        conn.execute("SELECT 1")            # здесь транзакция фактически стартует и создаёт снимок
        info["pid"] = conn.info.backend_pid
        started.set()                       # сигнал основному потоку: «транзакция открыта»
        # Вместо фиксированного time.sleep(2) ждём сигнала release (максимум 15 с): так
        # транзакция гарантированно остаётся открытой в момент проверки, как бы ни
        # отработали задержки. Остаётся она при этом «idle in transaction».
        release.wait(timeout=15)
    # Выход из with: COMMIT (нечего фиксировать) и закрытие соединения.

info, started, release = {}, threading.Event(), threading.Event()
t = threading.Thread(target=long_transaction, args=(info, started, release))
t.start()
started.wait(timeout=10)       # ждём, пока поток откроет транзакцию
time.sleep(1.5)                # даём ей «состариться» дольше порога в 1 секунду

print("\nПока долгая транзакция открыта (pid в потоке =", info.get("pid"), "):")
rows = find_long_transactions()
print_rows(rows)
print("   наша транзакция найдена:", any(r[0] == info.get("pid") for r in rows))

release.set()                  # отпускаем поток: транзакция завершается
t.join()
print("\nПосле завершения долгой транзакции:")
print_rows(find_long_transactions())

# КАКИЕ ЕЩЁ ПОЛЯ pg_stat_activity ПОЛЕЗНЫ ДЛЯ ДИАГНОСТИКИ БЛОКИРОВОК:
#  * wait_event_type / wait_event — НА ЧЁМ сессия ждёт прямо сейчас ('Lock' / 'transactionid'
#    означает ожидание блокировки другой транзакции; 'Client' — ждёт клиента);
#  * state_change — когда сессия последний раз сменила состояние (как давно она «простаивает
#    в транзакции»), а query_start — когда начал выполняться текущий/последний запрос;
#  * backend_xid — есть ли у транзакции XID, то есть писала ли она что-то (а значит, держит
#    блокировки строк); backend_xmin — держит ли она снимок и мешает ли VACUUM;
#  * usename, datname, application_name, client_addr — «чья» это сессия, из какой базы и с
#    какого хоста: нужно, чтобы найти и предупредить владельца;
#  * backend_start — когда соединение было открыто;
#  * функция pg_blocking_pids(pid) — какие сессии блокируют указанную (используется в
#    упражнении 8), и представление pg_locks — полный список блокировок.
# Особое внимание — state = 'idle in transaction': транзакция открыта, но клиент молчит.
# От таких сессий защищает параметр idle_in_transaction_session_timeout.

До запуска долгой транзакции:
   (долгих транзакций нет)

Пока долгая транзакция открыта (pid в потоке = 1714 ):
   (1714, datetime.timedelta(seconds=1, microseconds=514152), 'idle in transaction', '842', 'SELECT 1')
   наша транзакция найдена: True

После завершения долгой транзакции:
   (долгих транзакций нет)


## 3. Пессимистичные блокировки и SQLAlchemy

MVCC защищает от грязного чтения, но **не спасает от Lost Update** в бизнес-логике: два пользователя прочитали одно значение, оба записали — побеждает последний.

**Пессимистичный подход** — заблокировать строку до конца транзакции через `SELECT … FOR UPDATE`.

- `SELECT … FOR UPDATE` блокирует возвращённые строки до `COMMIT`/`ROLLBACK`.
- Другие транзакции, которые тоже запросят `FOR UPDATE` (или `UPDATE`/`DELETE`) на этих строках, будут ждать.
- Обычный `SELECT` без `FOR UPDATE` **не блокируется** — читатели не страдают.
- `FOR UPDATE` дополнительно заставляет транзакцию получить XID.

**Ключевые конструкции:**
- `SET LOCAL lock_timeout = '3s'` — таймаут ожидания блокировки только до конца текущей транзакции.
- `engine.begin()` — контекстный менеджер «соединение + транзакция»: `commit()` при успехе, `rollback()` при исключении.


In [30]:
from sqlalchemy.exc import OperationalError

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000)"))

def withdraw_money(account_id, amount):
    try:
        with engine.begin() as conn:
            conn.execute(text("SET LOCAL lock_timeout = '3s'"))
            row = conn.execute(
                text("SELECT balance FROM accounts WHERE id = :id FOR UPDATE"),
                {"id": account_id},
            ).fetchone()
            current = row[0]
            print(f"[pid-{account_id}] баланс={current}, списываем {amount}")
            if current >= amount:
                conn.execute(
                    text("UPDATE accounts SET balance = :b WHERE id = :id"),
                    {"b": current - amount, "id": account_id},
                )
                print("  списание успешно")
            else:
                print("  недостаточно средств")
    except OperationalError as e:
        print(f"[lock timeout] {type(e).__name__}: {str(e).splitlines()[0]}")

# Одиночный вызов — проходит без ожидания
withdraw_money(1, 100)

# Проверим состояние
with engine.connect() as conn:
    print("balance =", conn.execute(text("SELECT balance FROM accounts WHERE id=1")).scalar())


[pid-1] баланс=1000, списываем 100
  списание успешно
balance = 900


### Упражнение 6. Сериализация через `FOR UPDATE`

**Задание.** Убедитесь, что два параллельных `withdraw_money` выстраиваются в очередь, а не затирают друг друга.

**Шаги.**
1. Сбросьте `accounts`: `UPDATE accounts SET balance = 1000 WHERE id = 1`.
2. Запустите два потока, каждый вызывает `withdraw_money(1, 100)`. Оба должны завершиться **успешно** (баланс 1000 ≥ 200).
3. В конце проверьте: `SELECT balance FROM accounts WHERE id = 1` — должно быть **800** (1000 − 100 − 100). Это доказывает, что второй поток дождался первого и увидел уже обновлённый баланс.
4. Дополнительно: измените `withdraw_money` так, чтобы первый вызов держал транзакцию дольше (`time.sleep(4)` после `FOR UPDATE`). Второй поток должен упасть по `lock_timeout = '3s'`.
5. В комментарии: в каком бизнес-сценарии `FOR UPDATE` необходим, а в каком — избыточен?

**Критерий успеха.** При обычном запуске двух потоков баланс = 800. При искусственной задержке — второй падает по таймауту.


In [31]:
# УПРАЖНЕНИЕ 6. Сериализация конкурентных списаний через SELECT ... FOR UPDATE
#
# Что делаем: запускаем два параллельных списания с одного счёта и проверяем, что они
# выстраиваются в очередь (итог 800, а не 900), а при слишком долгом удержании блокировки
# второй поток падает по lock_timeout.
import threading, time
from sqlalchemy.exc import OperationalError

T0 = time.perf_counter()
def log(msg):
    """Печать с отметкой времени от начала эксперимента: видно, кто и сколько ждал."""
    print(f"[{time.perf_counter() - T0:4.1f}s] {msg}")

def get_balance():
    # engine.connect() — соединение из пула; транзакция неявная, на выходе из with соединение
    # возвращается в пул, а чтение ничего не меняет (откат безвреден).
    with engine.connect() as conn:
        return conn.execute(text("SELECT balance FROM accounts WHERE id = 1")).scalar()

def reset_balance(value=1000):
    # Upsert вместо голого UPDATE: ячейка работает, даже если accounts ещё не создана или была
    # пересоздана демо-ячейкой про deadlock (в ней другое содержимое).
    # engine.begin(): COMMIT при успехе, ROLLBACK при исключении.
    with engine.begin() as conn:
        conn.execute(text("CREATE TABLE IF NOT EXISTS accounts (id INT PRIMARY KEY, balance INT)"))
        conn.execute(
            text("INSERT INTO accounts (id, balance) VALUES (1, :b) "
                 "ON CONFLICT (id) DO UPDATE SET balance = EXCLUDED.balance"),
            {"b": value},
        )

def withdraw_money_ex6(account_id, amount, hold=0.0, who="T"):
    """withdraw_money из демо, но с параметром hold (искусственная задержка под блокировкой)
    и с метками потока who/временем в логе."""
    try:
        # Изоляция по умолчанию — READ COMMITTED. Именно поэтому FOR UPDATE здесь корректен:
        # после ожидания PostgreSQL перечитывает АКТУАЛЬНУЮ версию строки (см. вывод внизу).
        with engine.begin() as conn:
            # SET LOCAL действует до конца этой транзакции. Ждать блокировку дольше 3 с мы
            # не готовы: лучше упасть с понятной ошибкой, чем висеть бесконечно.
            conn.execute(text("SET LOCAL lock_timeout = '3s'"))
            log(f"{who}: просим блокировку строки счёта {account_id}")
            # FOR UPDATE: блокирует строку до конца транзакции. Если её уже держит другая
            # транзакция — ждём. Обычные SELECT без FOR UPDATE при этом не блокируются.
            row = conn.execute(
                text("SELECT balance FROM accounts WHERE id = :id FOR UPDATE"),
                {"id": account_id},
            ).fetchone()
            current = row[0]
            log(f"{who}: блокировка получена, баланс={current}, списываем {amount}")
            if hold:
                time.sleep(hold)           # имитация долгой бизнес-логики ПОД блокировкой
            if current >= amount:
                conn.execute(
                    text("UPDATE accounts SET balance = :b WHERE id = :id"),
                    {"b": current - amount, "id": account_id},
                )
                log(f"{who}: списание успешно (COMMIT на выходе из with)")
            else:
                log(f"{who}: недостаточно средств")
    except OperationalError as e:
        # psycopg.errors.LockNotAvailable (SQLSTATE 55P03) SQLAlchemy заворачивает в
        # OperationalError; оригинальное исключение драйвера лежит в e.orig.
        log(f"{who}: [lock timeout] {type(e.orig).__name__}: {str(e).splitlines()[0]}")

# --- Шаги 1-3. Два потока без искусственных задержек ----------------------------------
reset_balance(1000)
print("Баланс до запуска:", get_balance())
t1 = threading.Thread(target=withdraw_money_ex6, args=(1, 100), kwargs={"who": "T1"})
t2 = threading.Thread(target=withdraw_money_ex6, args=(1, 100), kwargs={"who": "T2"})
t1.start(); t2.start()
t1.join();  t2.join()
final_1 = get_balance()
print("Баланс после двух списаний:", final_1, "| ожидалось 800:", final_1 == 800)
# Почему 800, а не 900 (потерянное обновление): оба потока прочитали бы 1000 и записали бы 900,
# если бы читали без блокировки. FOR UPDATE заставляет второй поток ЖДАТЬ коммита первого, и
# после снятия блокировки он получает уже обновлённую строку (balance=900) и списывает с неё.

# --- Шаг 4. Первый поток держит блокировку 4 с, второй ждёт максимум 3 с ------------
print("\n--- Искусственная задержка: T1 держит блокировку 4 с, у T2 lock_timeout = 3 с ---")
T0 = time.perf_counter()              # обнуляем таймер для читаемого лога
reset_balance(1000)
t1 = threading.Thread(target=withdraw_money_ex6, args=(1, 100), kwargs={"hold": 4, "who": "T1"})
t2 = threading.Thread(target=withdraw_money_ex6, args=(1, 100), kwargs={"who": "T2"})
t1.start()
time.sleep(0.5)                       # гарантируем, что T1 первым возьмёт блокировку
t2.start()
t1.join();  t2.join()
final_2 = get_balance()
print("Баланс после эксперимента:", final_2,
      "| списал только T1 (900):", final_2 == 900)

# Критерий успеха: обычный запуск -> 800; при задержке T2 падает по таймауту (см. лог выше).

# В КАКОМ СЦЕНАРИИ FOR UPDATE НЕОБХОДИМ, А В КАКОМ ИЗБЫТОЧЕН:
# НЕОБХОДИМ, когда есть цикл «прочитать -> принять решение в приложении -> записать» и
# решение зависит от прочитанного значения, а конкурентные изменения недопустимы: списание
# и проверка достаточности средств, резерв товара на складе (остаток >= заказ), бронирование
# места или номера, выдача уникального номера по счётчику. Без блокировки два параллельных
# клиента принимают решение по одинаковому «устаревшему» значению (lost update).
# ИЗБЫТОЧЕН, когда:
#  * операция атомарна на стороне БД: UPDATE accounts SET balance = balance - 100
#    WHERE id = 1 AND balance >= 100 — UPDATE и сам берёт блокировку строки, и перечитывает её
#    после ожидания, отдельный FOR UPDATE ничего не добавляет;
#  * транзакция только читает (отчёты, карточки товара) — MVCC и так даёт согласованный снимок;
#  * конкуренции за одни строки почти нет, а повтор операции дёшев — лучше оптимистичный
#    подход (упражнение 7): блокировки сами по себе стоят ожиданий и риска deadlock;
#  * логика под блокировкой долгая (HTTP-вызов, ожидание пользователя): блокировка не должна
#    жить дольше нескольких миллисекунд, иначе очередь растёт, а транзакции становятся «долгими».
# Учтите: на REPEATABLE READ FOR UPDATE по строке, изменённой после снимка, вместо ожидания
# закончится ошибкой serialization failure, и приложение обязано повторить транзакцию.

Баланс до запуска: 1000
[ 0.0s] T1: просим блокировку строки счёта 1
[ 0.0s] T2: просим блокировку строки счёта 1
[ 0.0s] T1: блокировка получена, баланс=1000, списываем 100
[ 0.0s] T1: списание успешно (COMMIT на выходе из with)
[ 0.0s] T2: блокировка получена, баланс=900, списываем 100
[ 0.0s] T2: списание успешно (COMMIT на выходе из with)
Баланс после двух списаний: 800 | ожидалось 800: True

--- Искусственная задержка: T1 держит блокировку 4 с, у T2 lock_timeout = 3 с ---
[ 0.0s] T1: просим блокировку строки счёта 1
[ 0.0s] T1: блокировка получена, баланс=1000, списываем 100
[ 0.5s] T2: просим блокировку строки счёта 1
[ 3.5s] T2: [lock timeout] LockNotAvailable: (psycopg.errors.LockNotAvailable) canceling statement due to lock timeout
[ 4.0s] T1: списание успешно (COMMIT на выходе из with)
Баланс после эксперимента: 900 | списал только T1 (900): True


## 4. Оптимистичный контроль версий

Альтернатива пессимистичным блокировкам — **оптимистичная блокировка (OCC)**. Не блокируем строки, а добавляем колонку `version` (или используем `xmin`). При чтении запоминаем версию; `UPDATE` требует, чтобы версия не изменилась:

```sql
UPDATE seminar_tickets
SET status = 'booked', version = version + 1
WHERE id = :id AND version = :expected;
```

Если `rowcount = 0` — кто-то уже изменил запись. Приложение **повторяет** попытку (retry).

Плюсы:
- нет блокировок → выше пропускная способность при низкой конкуренции;
- хорошо работает в веб-приложениях с короткими транзакциями.

Минусы:
- при высокой конкуренции много retry;
- нужна дисциплина: не забыть добавить `WHERE version = ...`.

**Ключевая идея:** проверка версии в `WHERE` превращает «чтение-запись» в одну атомарную операцию. Lost Update невозможен даже без `FOR UPDATE`.


In [32]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS seminar_tickets"))
    conn.execute(text("""
        CREATE TABLE seminar_tickets (
            id INT PRIMARY KEY,
            status TEXT,
            version INT DEFAULT 0
        )
    """))
    conn.execute(text("INSERT INTO seminar_tickets (id, status) VALUES (1, 'free')"))

def book_ticket_optimistic(ticket_id, user_name, max_retries=3):
    for attempt in range(1, max_retries + 1):
        with engine.begin() as conn:
            row = conn.execute(
                text("SELECT status, version FROM seminar_tickets WHERE id = :id"),
                {"id": ticket_id},
            ).fetchone()
            if row is None:
                print(f"{user_name}: билет не существует")
                return
            status, current_version = row
            if status != 'free':
                print(f"{user_name}: билет уже занят ({status})")
                return
            res = conn.execute(
                text("""
                    UPDATE seminar_tickets
                    SET status = :s, version = version + 1
                    WHERE id = :id AND version = :v
                """),
                {"s": f"booked_by_{user_name}", "id": ticket_id, "v": current_version},
            )
            if res.rowcount == 0:
                print(f"{user_name}: конфликт версий, попытка {attempt}/{max_retries}")
                continue
            print(f"{user_name}: успешно забронировал")
            return
    print(f"{user_name}: не удалось за {max_retries} попыток")

book_ticket_optimistic(1, "alice")


alice: успешно забронировал


### Упражнение 7. Гонка за билетом

**Задание.** Запустите 5 потоков, пытающихся одновременно забронировать один и тот же билет. Только один должен добиться успеха, остальные — увидеть «билет уже занят» (возможно, после retry).

**Шаги.**
1. Сбросьте `seminar_tickets`: `UPDATE seminar_tickets SET status='free', version=0 WHERE id=1`.
2. Создайте 5 потоков, каждый вызывает `book_ticket_optimistic(1, f"user_{i}")`.
3. Запустите все через `threading.Thread`, дождитесь `join()`.
4. Проверьте: `SELECT status, version FROM seminar_tickets WHERE id=1` — статус = `booked_by_user_X`, `version ≥ 1`.
5. Дополнительно: посчитайте, сколько успешных бронирований и сколько конфликтов.
6. В комментарии: почему оптимистичный подход даёт лучшую пропускную способность при низкой конкуренции, но хуже — при высокой?

**Критерий успеха.** Ровно 1 успешная бронь. В консоли видны `конфликт версий` и `билет уже занят` от остальных.


In [33]:
# УПРАЖНЕНИЕ 7. Гонка за билетом: оптимистичная блокировка при 5 конкурентах
#
# Что делаем: 5 потоков одновременно пытаются забронировать один билет. Ожидаем ровно одну
# успешную бронь, а остальные должны получить «конфликт версий» и после повтора — «билет занят».
import threading, time

# --- Шаг 1. Сброс состояния -----------------------------------------------------------
# Upsert: ячейка работает и после демо (в нём билет уже забронирован alice), и при запуске
# отдельно (таблицу создаём, если её нет). Всё в одной транзакции (engine.begin()).
def reset_ticket():
    with engine.begin() as conn:
        conn.execute(text(
            "CREATE TABLE IF NOT EXISTS seminar_tickets "
            "(id INT PRIMARY KEY, status TEXT, version INT DEFAULT 0)"
        ))
        conn.execute(text(
            "INSERT INTO seminar_tickets (id, status, version) VALUES (1, 'free', 0) "
            "ON CONFLICT (id) DO UPDATE SET status = 'free', version = 0"
        ))

# Счётчики итогов. Доступ из нескольких потоков идёт под Lock: операция += у int в Python
# не атомарна, без блокировки счётчик мог бы «потерять» часть увеличений (тот же lost update).
stats = {"success": 0, "conflicts": 0, "already_taken": 0, "gave_up": 0}
stats_lock = threading.Lock()

def bump(key):
    with stats_lock:
        stats[key] += 1

def book_ticket_counted(ticket_id, user_name, start_barrier, think=0.2, max_retries=3):
    """Копия book_ticket_optimistic из демо, дополненная подсчётом исходов и паузой think."""
    # Barrier(5): все 5 потоков ждут друг друга и стартуют ОДНОВРЕМЕННО. Без этого поток
    # успевал бы забронировать билет раньше, чем запустится следующий, и конфликтов не было бы.
    start_barrier.wait()
    for attempt in range(1, max_retries + 1):
        # Каждая попытка — НОВАЯ короткая транзакция (READ COMMITTED по умолчанию), чтобы
        # на повторе мы увидели свежее состояние билета.
        with engine.begin() as conn:
            row = conn.execute(
                text("SELECT status, version FROM seminar_tickets WHERE id = :id"),
                {"id": ticket_id},
            ).fetchone()
            if row is None:
                print(f"{user_name}: билет не существует")
                return
            status, current_version = row
            if status != "free":
                print(f"{user_name}: билет уже занят ({status})")
                bump("already_taken")
                return

            # Пауза между чтением и записью имитирует «пользователь думает / идёт бизнес-логика».
            # Она гарантирует, что все 5 потоков успеют прочитать (free, version=0) ДО чьей-либо
            # записи, то есть гонка реально состоится. Блокировок на это время нет: читатели
            # друг другу не мешают.
            time.sleep(think)

            # Условная запись: версия проверяется в WHERE, то есть «проверить и записать» —
            # одна атомарная операция. Первый UPDATE берёт блокировку строки; остальные ждут её
            # снятия, затем (в READ COMMITTED) PostgreSQL перепроверяет WHERE на НОВОЙ версии
            # строки: там version = 1, а мы ждём 0 -> подходящих строк нет -> rowcount = 0.
            res = conn.execute(
                text("""
                    UPDATE seminar_tickets
                    SET status = :s, version = version + 1
                    WHERE id = :id AND version = :v
                """),
                {"s": f"booked_by_{user_name}", "id": ticket_id, "v": current_version},
            )
            if res.rowcount == 0:
                print(f"{user_name}: конфликт версий, попытка {attempt}/{max_retries}")
                bump("conflicts")
                continue          # выходим из with (COMMIT пустой транзакции) и пробуем снова
            print(f"{user_name}: успешно забронировал")
            bump("success")
            return
    print(f"{user_name}: не удалось за {max_retries} попыток")
    bump("gave_up")

# --- Шаги 2-3. Запуск 5 потоков ----------------------------------------------------------
reset_ticket()
N_THREADS = 5
barrier = threading.Barrier(N_THREADS)
threads = [
    threading.Thread(target=book_ticket_counted, args=(1, f"user_{i}", barrier))
    for i in range(N_THREADS)
]
for th in threads:
    th.start()
for th in threads:
    th.join()                     # ждём все потоки

# --- Шаги 4-5. Проверка итогов ------------------------------------------------------------
with engine.connect() as conn:
    status, version = conn.execute(
        text("SELECT status, version FROM seminar_tickets WHERE id = 1")
    ).one()
print("\nИтоговое состояние билета:", status, "| version =", version)
print("Статистика:", stats)
print(f"  успешных броней  : {stats['success']}")
print(f"  конфликтов версий: {stats['conflicts']}")
print(f"  «уже занят»      : {stats['already_taken']}")
print("Критерий: ровно 1 успех ->", stats["success"] == 1,
      "| version >= 1 ->", version >= 1,
      "| статус booked_by_user_X ->", status.startswith("booked_by_user_"))

# ПОЧЕМУ ОПТИМИСТИЧНЫЙ ПОДХОД ЛУЧШЕ ПРИ НИЗКОЙ КОНКУРЕНЦИИ И ХУЖЕ ПРИ ВЫСОКОЙ:
# * Низкая конкуренция: за одну строку почти никто не борется, поэтому проверка версии почти
#   всегда проходит с первого раза. Блокировок на время «чтения и размышлений» нет, соединения
#   не стоят в очередях, deadlock невозможен, а лишних операций ровно одна — условный UPDATE.
#   Пропускная способность выше, чем у FOR UPDATE, где каждая транзакция удерживает строку
#   (и стоит в очереди) от чтения до коммита.
# * Высокая конкуренция: как в нашем эксперименте, из 5 попыток 4 завершаются конфликтом.
#   Проигравшие уже выполнили чтение и бизнес-логику, и вся эта работа пропадает; потом они
#   повторяют её с нуля, снова мешая друг другу. Нагрузка на БД растёт, задержка растёт,
#   возможен «голод» (поток, который не может выиграть за max_retries попыток). Пессимистичная
#   блокировка в этом случае выгоднее: все работают в очереди и каждый делает работу один раз.
# * Итого: оптимистичный подход ставит на редкие конфликты (повтор дешёвый), пессимистичный —
#   на частые (повтор дорогой, проще подождать). Выбор определяется ожидаемой частотой конфликтов.

user_4: успешно забронировал
user_1: конфликт версий, попытка 1/3
user_0: конфликт версий, попытка 1/3
user_2: конфликт версий, попытка 1/3
user_3: конфликт версий, попытка 1/3
user_1: билет уже занят (booked_by_user_4)
user_0: билет уже занят (booked_by_user_4)
user_3: билет уже занят (booked_by_user_4)
user_2: билет уже занят (booked_by_user_4)

Итоговое состояние билета: booked_by_user_4 | version = 1
Статистика: {'success': 1, 'conflicts': 4, 'already_taken': 4, 'gave_up': 0}
  успешных броней  : 1
  конфликтов версий: 4
  «уже занят»      : 4
Критерий: ровно 1 успех -> True | version >= 1 -> True | статус booked_by_user_X -> True


## 5. Deadlock (взаимная блокировка)

**Deadlock** — Транзакция A держит Ресурс 1 и ждёт Ресурс 2; Транзакция B держит Ресурс 2 и ждёт Ресурс 1. Цикл.

PostgreSQL обнаруживает цикл через **`deadlock_timeout`** (по умолчанию **1 секунда**): если транзакция ждёт блокировку дольше — строится граф ожиданий, ищется цикл. При обнаружении одна из транзакций откатывается с ошибкой `deadlock detected`.

Обратите внимание: `deadlock_timeout` (1 с) обычно **меньше** `lock_timeout` (например, 3 с), поэтому при deadlock вы увидите именно `DeadlockDetected`, а не `LockNotAvailable`.

**Как лечить:**
- Единый порядок захвата блокировок во всех транзакциях.
- Короткие транзакции.
- Обработка `OperationalError` с retry.

**Как диагностировать:**
- `pg_stat_activity` + `pg_blocking_pids(pid)` — кто кого держит.
- `pg_locks` — все блокировки.
- `SELECT pg_terminate_backend(<blocking_pid>)` — точечно завершить «виновника».


In [34]:
import threading, time
from sqlalchemy.exc import OperationalError

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000), (2, 2000)"))

def transaction_1():
    try:
        with engine.begin() as conn:
            print("T1: блокируем счёт 1")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            time.sleep(1)
            print("T1: пытаемся заблокировать счёт 2...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            print("T1: успех")
    except OperationalError as e:
        print(f"T1 повалена: {type(e).__name__}: {str(e).splitlines()[0]}")

def transaction_2():
    try:
        with engine.begin() as conn:
            print("T2: блокируем счёт 2")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            time.sleep(1)
            print("T2: пытаемся заблокировать счёт 1...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            print("T2: успех")
    except OperationalError as e:
        print(f"T2 повалена: {type(e).__name__}: {str(e).splitlines()[0]}")

t1 = threading.Thread(target=transaction_1)
t2 = threading.Thread(target=transaction_2)
t1.start(); t2.start()
t1.join(); t2.join()


T1: блокируем счёт 1
T2: блокируем счёт 2
T1: пытаемся заблокировать счёт 2...
T2: пытаемся заблокировать счёт 1...
T2: успех
T1 повалена: OperationalError: (psycopg.errors.DeadlockDetected) deadlock detected


### Упражнение 8. Устраняем deadlock

**Задание.** «Вылечите» демонстрацию: измените `transaction_2` так, чтобы она блокировала счета **в том же порядке**, что и `transaction_1` (сначала счёт 1, потом счёт 2). Deadlock должен исчезнуть.

**Шаги.**
1. Скопируйте демо-код в новую ячейку.
2. Измените `transaction_2`: вместо блокировки счёта 2 первым шагом блокируйте счёт 1.
3. Запустите скрипт **несколько раз** (3–5). Deadlock не должен появляться.
4. В комментарии объясните, почему единый порядок захвата ресурсов делает deadlock невозможным.
5. Дополнительно: напишите запрос `SELECT blocked.pid, blocking.pid, blocked.query FROM ...` с использованием `pg_blocking_pids(blocked.pid)` — выведите, кто кого блокирует, если один из потоков висит.

**Критерий успеха.** Обе транзакции всегда печатают `успех`. Ни в одном прогоне нет `DeadlockDetected`.


In [35]:
# УПРАЖНЕНИЕ 8. Устраняем deadlock единым порядком блокировок
#
# Что делаем: оставляем transaction_1 как есть (счёт 1, потом счёт 2) и исправляем
# transaction_2 так, чтобы она тоже брала счета в порядке 1 -> 2. Прогоняем 5 раз и убеждаемся,
# что deadlock не возникает. Затем показываем диагностику «кто кого блокирует».
import threading, time
from sqlalchemy.exc import OperationalError

# Свежие данные: два счёта (та же таблица, что в демо; пересоздание нужно, чтобы ячейка
# не зависела от того, что делали упражнения 6 и 7).
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000), (2, 2000)"))

def transaction_1(out):
    """Без изменений: блокирует счёт 1, затем счёт 2. Итог пишет в словарь out."""
    try:
        # engine.begin(): одна транзакция; все блокировки FOR UPDATE снимаются на выходе из with
        # (COMMIT при успехе или ROLLBACK при ошибке).
        with engine.begin() as conn:
            print("T1: блокируем счёт 1")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            time.sleep(1)       # «работа» под блокировкой; в демо именно здесь второй поток успевал взять счёт 2
            print("T1: пытаемся заблокировать счёт 2...")
            # lock_timeout ограничивает ожидание: даже без deadlock мы не зависнем навсегда.
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            print("T1: успех")
            out["T1"] = "успех"
    except OperationalError as e:
        # e.orig — исходное исключение драйвера (DeadlockDetected или LockNotAvailable);
        # у обёртки SQLAlchemy тип всегда просто OperationalError.
        print(f"T1 повалена: {type(e.orig).__name__}: {str(e).splitlines()[0]}")
        out["T1"] = type(e.orig).__name__

def transaction_2_fixed(out):
    """ИСПРАВЛЕНО: теперь тоже сначала счёт 1, потом счёт 2 — тот же порядок, что у T1."""
    try:
        with engine.begin() as conn:
            # Было: сначала счёт 2, потом счёт 1 (порядок, обратный T1) — это и давало цикл.
            # Стало: тот же порядок 1 -> 2. Теперь тот, кто не успел, будет просто ЖДАТЬ
            # первый же ресурс, не удерживая ничего нужного другому.
            print("T2: блокируем счёт 1")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            time.sleep(1)
            print("T2: пытаемся заблокировать счёт 2...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            print("T2: успех")
            out["T2"] = "успех"
    except OperationalError as e:
        print(f"T2 повалена: {type(e.orig).__name__}: {str(e).splitlines()[0]}")
        out["T2"] = type(e.orig).__name__

# --- Шаг 3. Запускаем 5 раз -----------------------------------------------------------------
# Один из потоков первым получит счёт 1, второй будет ждать его в первом же SELECT (без
# таймаута: ожидание короткое, пока первый не закоммитится, а он ничего не ждёт). Кто победит,
# от запуска к запуску случайно, на результат это не влияет.
runs = []
for run in range(1, 6):
    print(f"\n=== Прогон {run} ===")
    out = {}
    t1 = threading.Thread(target=transaction_1, args=(out,))
    t2 = threading.Thread(target=transaction_2_fixed, args=(out,))
    t1.start(); t2.start()
    t1.join();  t2.join()
    runs.append(out)

print("\nИтоги прогонов:", runs)
print("Критерий: во всех прогонах обе транзакции — «успех»:",
      all(r.get("T1") == "успех" and r.get("T2") == "успех" for r in runs))
print("Ни одного DeadlockDetected:",
      not any("DeadlockDetected" in r.values() for r in runs))

# ПОЧЕМУ ЕДИНЫЙ ПОРЯДОК ЗАХВАТА ДЕЛАЕТ DEADLOCK НЕВОЗМОЖНЫМ:
# Deadlock — это цикл в графе ожиданий: T1 ждёт T2, T2 ждёт ... и снова T1. Пусть все
# транзакции берут ресурсы строго по возрастанию номера (1, затем 2, ...). Для транзакции T
# обозначим m(T) наибольший номер ресурса, который она уже держит. Когда T ждёт ресурс r,
# мы знаем, что r > m(T) (иначе T запрашивала бы его раньше), а держит r другая транзакция T',
# значит, m(T') >= r > m(T). Идя по цепочке «кто кого ждёт», m строго растёт и никогда не
# вернётся к исходному значению — замкнуть цикл невозможно. В исходном демо T2 брала 2, а
# потом 1, нарушала порядок, и это давало цикл: T1 держит 1 и ждёт 2, T2 держит 2 и ждёт 1.
# Практическое правило: если ресурсов несколько, сортируйте их до блокировки, например
# SELECT ... WHERE id IN (...) ORDER BY id FOR UPDATE.

# --- Шаг 5 (дополнительно). Диагностика: кто кого блокирует -----------------------------
# Чтобы получить «зависший» поток, делаем условный сценарий: holder берёт блокировку счёта 1
# и держит, а waiter пытается взять её же и ждёт. Пока они стоят, смотрим pg_blocking_pids.
BLOCKING_SQL = """
    SELECT blocked.pid                  AS blocked_pid,
           blocking.pid                 AS blocking_pid,
           blocked.state                AS blocked_state,
           blocked.wait_event_type      AS wait_type,
           blocked.wait_event           AS wait_event,
           left(blocked.query, 55)      AS blocked_query,
           left(blocking.query, 55)     AS blocking_query
    FROM pg_stat_activity AS blocked
    JOIN pg_stat_activity AS blocking
      ON blocking.pid = ANY(pg_blocking_pids(blocked.pid))
"""
# pg_blocking_pids(pid) возвращает МАССИВ pid сессий, которые блокируют сессию pid
# (пустой массив, если не блокирует никто), поэтому соединяем через = ANY(массив).

def holder(started, release):
    with engine.begin() as conn:
        conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
        started.set()                     # «блокировка взята»
        release.wait(timeout=15)          # держим до сигнала; затем COMMIT снимает блокировку

def waiter():
    with engine.begin() as conn:
        conn.execute(text("SET LOCAL lock_timeout = '10s'"))
        conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))   # встанет в очередь

started, release = threading.Event(), threading.Event()
th_h = threading.Thread(target=holder, args=(started, release))
th_w = threading.Thread(target=waiter)
th_h.start(); started.wait(timeout=10)
th_w.start()

# Ждём (до ~5 с), пока waiter реально упрётся в блокировку, и смотрим диагностику.
rows = []
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as mon:   # autocommit: свежий снимок статистики на каждый запрос
    for _ in range(20):
        rows = mon.execute(BLOCKING_SQL).fetchall()
        if rows:
            break
        time.sleep(0.25)
print("\nКто кого блокирует (blocked_pid, blocking_pid, state, wait_type, wait_event, запросы):")
for r in rows:
    print("  ", r)
if not rows:
    print("   (блокировок не найдено)")

release.set()                             # holder делает COMMIT, waiter получает блокировку и завершается
th_h.join(); th_w.join()
# Если «виновник» завис по-настоящему, его можно завершить точечно:
#   SELECT pg_terminate_backend(<blocking_pid>);
# Сессию оборвут, транзакция откатится, блокировки снимутся. Применять надо осознанно.


=== Прогон 1 ===
T1: блокируем счёт 1
T2: блокируем счёт 1
T1: пытаемся заблокировать счёт 2...
T1: успех
T2: пытаемся заблокировать счёт 2...
T2: успех

=== Прогон 2 ===
T1: блокируем счёт 1
T2: блокируем счёт 1
T1: пытаемся заблокировать счёт 2...
T1: успех
T2: пытаемся заблокировать счёт 2...
T2: успех

=== Прогон 3 ===
T1: блокируем счёт 1
T2: блокируем счёт 1
T2: пытаемся заблокировать счёт 2...
T2: успех
T1: пытаемся заблокировать счёт 2...
T1: успех

=== Прогон 4 ===
T1: блокируем счёт 1
T2: блокируем счёт 1
T2: пытаемся заблокировать счёт 2...
T2: успех
T1: пытаемся заблокировать счёт 2...
T1: успех

=== Прогон 5 ===
T1: блокируем счёт 1
T2: блокируем счёт 1
T2: пытаемся заблокировать счёт 2...
T2: успех
T1: пытаемся заблокировать счёт 2...
T1: успех

Итоги прогонов: [{'T1': 'успех', 'T2': 'успех'}, {'T1': 'успех', 'T2': 'успех'}, {'T2': 'успех', 'T1': 'успех'}, {'T2': 'успех', 'T1': 'успех'}, {'T2': 'успех', 'T1': 'успех'}]
Критерий: во всех прогонах обе транзакции — «успех»: